# Analyze an Invoice with Azure AI Document Intelligence

Run these cells one at a time to analyze a sample invoice with the prebuilt invoice model. The notebook uses `DOC_INTELLIGENCE_KEY` when it is set, and falls back to your current Azure user credentials when the key is empty.

## Before You Start

Make sure you have installed the shared workshop requirements and copied `workshop/.env.sample` to `workshop/.env`. Set `DOC_INTELLIGENCE_ENDPOINT` in `workshop/.env`. You can leave `DOC_INTELLIGENCE_KEY` empty if your current Azure user has access to the Document Intelligence resource.

In [2]:
from pathlib import Path
import os

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

## Load Configuration

This cell finds `workshop/.env` from either the repository root or the notebook folder, then reads the Document Intelligence endpoint and optional key.

In [3]:
def find_workshop_env() -> Path:
    start = Path.cwd().resolve()
    candidates = [start / 'workshop' / '.env', start / '.env']

    for parent in start.parents:
        candidates.extend([parent / 'workshop' / '.env', parent / '.env'])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError('Could not find workshop/.env. Copy workshop/.env.sample to workshop/.env first.')


env_path = find_workshop_env()
load_dotenv(env_path)

endpoint = (os.getenv('DOC_INTELLIGENCE_ENDPOINT') or '').strip()
key = (os.getenv('DOC_INTELLIGENCE_KEY') or '').strip()

if not endpoint:
    raise ValueError('DOC_INTELLIGENCE_ENDPOINT is not set in workshop/.env.')

print(f'Loaded settings from: {env_path}')
print(f'Document Intelligence endpoint: {endpoint}')
print('Authentication: key credential' if key else 'Authentication: current Azure user credentials')

Loaded settings from: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Document Intelligence endpoint: https://docintelltecjexcelaiai.cognitiveservices.azure.com/
Authentication: current Azure user credentials


## Set Analysis Inputs

The sample invoice is hosted in the source lab repository. The model ID uses the built-in invoice model.

In [4]:
file_uri = 'https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/03-document-intelligence/prebuilt/sample-invoice/sample-invoice.pdf?raw=true'
file_locale = 'en-US'
file_model_id = 'prebuilt-invoice'

print(f'Analyzing invoice at: {file_uri}')
print(f'Using model: {file_model_id}')

Analyzing invoice at: https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/03-document-intelligence/prebuilt/sample-invoice/sample-invoice.pdf?raw=true
Using model: prebuilt-invoice


## Create the Client

If `DOC_INTELLIGENCE_KEY` has a value, the client uses key authentication. If the key is blank, the client uses `DefaultAzureCredential`, which can use your current Azure CLI, VS Code, managed identity, or other configured Azure identity.

In [5]:
credential = AzureKeyCredential(key) if key else DefaultAzureCredential()
document_analysis_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential
)

print('Document Intelligence client created.')

Document Intelligence client created.


## Analyze the Invoice

This cell sends the invoice URL to Document Intelligence and waits for the operation to complete.

In [6]:
poller = document_analysis_client.begin_analyze_document(
    file_model_id,
    AnalyzeDocumentRequest(url_source=file_uri),
    locale=file_locale
)

result = poller.result()
print('Analysis complete.')

Analysis complete.


## Review Extracted Fields

The prebuilt invoice model returns structured fields such as vendor name, customer name, and invoice total.

In [7]:
for document in result.documents:
    vendor_name = document.fields.get('VendorName')
    if vendor_name:
        print(f"Vendor Name: {vendor_name.get('valueString')}, with confidence {vendor_name.get('confidence')}.")

    customer_name = document.fields.get('CustomerName')
    if customer_name:
        print(f"Customer Name: {customer_name.get('valueString')}, with confidence {customer_name.get('confidence')}.")

    invoice_total = document.fields.get('InvoiceTotal')
    if invoice_total:
        amount = invoice_total.get('valueCurrency', {})
        currency_symbol = amount.get('currencySymbol', '$')
        value = amount.get('amount')
        print(f"Invoice Total: {currency_symbol}{value}, with confidence {invoice_total.get('confidence')}.")

Vendor Name: CONTOSO LTD., with confidence 0.936.
Customer Name: MICROSOFT CORPORATION, with confidence 0.891.
Invoice Total: $110.0, with confidence 0.939.


## Optional: Inspect All Returned Fields

Run this cell if you want to see every field returned by the prebuilt invoice model.

In [8]:
for document_index, document in enumerate(result.documents, start=1):
    print(f'Document {document_index}')
    for field_name, field in document.fields.items():
        value = field.get('content') or field.get('valueString') or field.get('valueCurrency') or field.get('valueDate')
        print(f'- {field_name}: {value} (confidence: {field.get("confidence")})')

Document 1
- AmountDue: $610.00 (confidence: 0.818)
- BillingAddress: 123 Bill St,
Redmond WA, 98052 (confidence: 0.889)
- BillingAddressRecipient: Microsoft Finance (confidence: 0.938)
- CustomerAddress: 123 Other St,
Redmond WA, 98052 (confidence: 0.875)
- CustomerAddressRecipient: Microsoft Corp (confidence: 0.889)
- CustomerId: CID-12345 (confidence: 0.956)
- CustomerName: MICROSOFT CORPORATION (confidence: 0.891)
- DueDate: 12/15/2019 (confidence: 0.971)
- InvoiceDate: 11/15/2019 (confidence: 0.971)
- InvoiceId: INV-100 (confidence: 0.971)
- InvoiceTotal: $110.00 (confidence: 0.939)
- Items: None (confidence: None)
- PreviousUnpaidBalance: $500.00 (confidence: 0.927)
- PurchaseOrder: PO-3333 (confidence: 0.966)
- RemittanceAddress: 123 Remit St
New York, NY, 10001 (confidence: 0.887)
- RemittanceAddressRecipient: Contoso Billing (confidence: 0.938)
- ServiceAddress: 123 Service St,
Redmond WA, 98052 (confidence: 0.887)
- ServiceAddressRecipient: Microsoft Services (confidence: 0.9